Code to go through Databricks schema and test if the views are working 


In [0]:

%sql
select * from system.information_schema.views where table_schema = 'gold'

In [0]:
%python
from pyspark.errors import PySparkException

df_list = (
    spark.sql(
        "select * from system.information_schema.views where table_schema = 'gold'"
    )
    .select("table_name", "view_definition")
    .filter("table_name like '%t%'")
    .collect()
   

)

In [0]:
%python
 df_list_sorted = sorted(df_list, key=lambda x: x["table_name"], reverse=False)
 display(df_list_sorted)

In [0]:
%python
for index, table_name in enumerate(df_list_sorted):
    testme = spark.sql(f"DESCRIBE f1_dev.gold.{table_name['table_name']}").collect()
    try:
        if index < 50:  #limit result set becuase of memory issues
            print(f"Index: {index} - {table_name['table_name']}")
            spark.sql(table_name['view_definition']).show(1)
           
    except PySparkException as ex:
        print("Error Class : " + ex.getErrorClass())
        print("Message parameters: " + str(ex.getMessageParameters()))